In [ ]:
import os
import requests
import pandas as pd
import random
import time
import urllib3
from pathlib import Path
from evidently import Report
from evidently.presets import DataDriftPreset
from IPython.display import IFrame

In [ ]:
URL = "https://$your_url/v2/models/$model/infer"
TOKEN = "$your_token"


In [ ]:
os.makedirs("data/reports", exist_ok=True)

DATA_DIR = Path("./data")
REPORT_DIR = DATA_DIR / "reports"

REFERENCE_PATH = DATA_DIR / "reference_data.csv"
CURRENT_PATH = DATA_DIR / "current_data.csv"
REPORT_PATH = REPORT_DIR / "iris_drift_report.html"

In [ ]:
reference = pd.read_csv(REFERENCE_PATH)
reference.head()

In [ ]:
# Suppress warnings because this demo connects to an internal
# OpenShift route using a self-signed/internal CA certificate.
urllib3.disable_warnings()

rows = []

for i in range(200):
    # intentionally drifted iris-like data
    sample = [
        random.uniform(6.5, 8.0),   # sepal length shifted higher
        random.uniform(2.0, 3.0),   # sepal width shifted lower
        random.uniform(4.5, 6.8),   # petal length shifted higher
        random.uniform(1.5, 2.5),
    ]
    payload = {
        "inputs": [{
            "name": "predict",
            "shape": [1, 4],
            "datatype": "FP32",
            "data": [sample]
        }]
    }
    r = requests.post(
        URL,
        headers={
            "Authorization": f"Bearer {TOKEN}",
            "Content-Type": "application/json"
        },
        json=payload,
        verify=False
    )

    result = r.json()
    
    prediction = result["outputs"][0]["data"][0]

    rows.append({
        "sepal_length": sample[0],
        "sepal_width": sample[1],
        "petal_length": sample[2],
        "petal_width": sample[3],
        "prediction": prediction
    })

    time.sleep(0.1)

current = pd.DataFrame(rows)
current = current.round({
    "sepal_length": 2,
    "sepal_width": 2,
    "petal_length": 2,
    "petal_width": 1
})
current.to_csv(CURRENT_PATH, index=False)

current.head()


In [ ]:
current.describe()

In [ ]:
print(reference.columns.tolist())
print(current.columns.tolist())

In [ ]:
reference = reference.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
})

FEATURES = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
]

reference_features = reference[FEATURES]
current_features = current[FEATURES]

In [ ]:
report = Report([
    DataDriftPreset()
])
my_eval = report.run(    
    reference_data=reference,
    current_data=current
)

my_eval.save_html(str(REPORT_PATH))
print("Exists:", REPORT_PATH.exists())
print("Path:", REPORT_PATH.resolve())

In [ ]:
result_dict = my_eval.dict()
result_dict


In [ ]:
IFrame(str(REPORT_PATH), width=1200, height=800)